Good call — that's the lower-risk path and it matches the failure mode. Here are the concrete knobs in your code, in the order I'd turn them.

The two cheapest changes target overfitting directly and are one line each. Bump fusion dropout and weight decay where you instantiate the model and optimiser in the training cell:

```python
model = HybridEHRModel(
    n_features=N_FEATURES, n_labels=N_LABELS,
    dropout_fusion=0.4,          # was 0.2
).to(DEVICE)

optimiser = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=1e-3,    # was 1e-4
)
```

Both push the same direction (less effective capacity), so I'd treat them as one "more regularization" experiment rather than two. `0.4` dropout and `1e-3` decay are moderate, safe starting points; if it helps but you're still seeing the train/val divergence, push to `0.5` and `1e-2`.

If that isn't enough, the structural lever is shrinking the shared embedding, which cuts capacity in both branches and the fusion layer at once:

```python
model = HybridEHRModel(
    n_features=N_FEATURES, n_labels=N_LABELS,
    embed_dim=128,               # was 256 → projection becomes 1024→128, fusion 256
    dropout_fusion=0.4,
).to(DEVICE)
```

Going 256→128 roughly halves the trainable parameters in the projection head and classifier, which is the most direct way to "trim" them. You could also raise the projection head's own dropout, but it's currently hardcoded to `0.1` inside `OpenMedEncoder` and not threaded through `HybridEHRModel`. Rather than edit the default, pass it through so it's tunable:

```python
# in HybridEHRModel.__init__, where text_branch is built:
self.text_branch = OpenMedEncoder(openmed_base_model or openmed_base,
                                  proj_dim=embed_dim, dropout=dropout_text)
# and add dropout_text: float = 0.1 to the __init__ signature
```

Then `dropout_text=0.3` becomes a knob like the others.

Two things on method, because they matter more than which knob you pick. Change one direction at a time and judge each on the **5-fold CV mean ± std**, not fold 0 — recall a ~0.005 AUC move on 276 patients is inside the noise, so a single fold can't tell you whether a knob helped. And more regularization slows convergence, so don't cut `N_EPOCHS`; your early stopping already captures the best epoch, so the only cost of leaving it at 30 is compute. If the best epoch starts landing near epoch 30 after you regularize, bump `N_EPOCHS` up rather than down.

One expectation to set: regularization rarely moves the headline number much on a cohort this size — it mostly stabilizes the gap between train and val. If your goal is a higher absolute AUC, the data volume is still the binding constraint, and these changes are about making the 0.85 you have trustworthy and reproducible across folds rather than pushing it to 0.88.